This file is to experiment with setting thresholds for our data with NLP-produced similarity scores
- for now, just using some sample file(s) from get_scores()
- sectioning off data by each .1 of similarity scores (.9-1.0, .8-.9, etc)
- then, get cohen's kappas on that for comparison

In [41]:
import pandas as pd
from NLP_Eval_for_DE import scores, data
import ast
from sklearn.metrics import cohen_kappa_score

In [ ]:
testable_data = data.get_testable_data("Example\\inputs\\case study 1 input\\pain points full.csv")
codes = data.get_codes("Example\\inputs\\case study 1 input\\short titles.csv")
all_scores = scores.get_Jina_scores(testable_data, codes)[1]

#split the lots of scores in the "Similarity scores" column into separate columns
all_scores_expanded = all_scores.copy()
all_scores_expanded[["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]] = pd.DataFrame(all_scores_expanded["Similarity scores"].tolist(), index=all_scores_expanded.index)
# add the "Consensus code" column of testable_data to all_scores_expanded
all_scores_expanded["Consensus code"] = testable_data["Consensus code"].tolist()
#delete the "Similarity scores" column
all_scores_expanded = all_scores_expanded.drop(columns=["Similarity scores"])
all_scores_expanded

,Input phrase,Similarity scores,1,2,3,4,5,6,7,8,9,10,11,Consensus code
0,cutting wood,"[0.65914696, 0.68691725, 0.6393915, 0.7534347,...",0.659147,0.686917,0.639391,0.753435,0.718065,0.797052,0.696077,0.789611,0.714689,0.691254,0.618896,0
1,didn�t know how to use lathe,"[0.7758371, 0.7071324, 0.5981057, 0.65807265, ...",0.775837,0.707132,0.598106,0.658073,0.673782,0.651082,0.650906,0.666729,0.711350,0.613773,0.586010,1
2,Finding drill,"[0.7039082, 0.82575095, 0.64815205, 0.6671239,...",0.703908,0.825751,0.648152,0.667124,0.709036,0.691553,0.717402,0.629175,0.824288,0.637896,0.636742,9
3,Taking out trash,"[0.61731094, 0.65654826, 0.639607, 0.63682663,...",0.617311,0.656548,0.639607,0.636827,0.704794,0.726124,0.715366,0.743981,0.678400,0.816364,0.599845,10
4,Finding clamp,"[0.698143, 0.67444193, 0.62876827, 0.6766978, ...",0.698143,0.674442,0.628768,0.676698,0.885455,0.673837,0.720935,0.655932,0.699091,0.689412,0.633356,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
394,Incorrect size gloves,"[0.6595968, 0.7382914, 0.8386486, 0.6418756, 0...",0.659597,0.738291,0.838649,0.641876,0.643478,0.623995,0.645385,0.623544,0.649213,0.657648,0.595388,3
395,Uncleaned machines from previous users,"[0.66961616, 0.6212497, 0.6044808, 0.6145736, ...",0.669616,0.621250,0.604481,0.614574,0.720937,0.663420,0.735110,0.704761,0.707858,0.674458,0.597780,6
396,Machine incorrectly set up by previous user,"[0.7067125, 0.67998904, 0.6040233, 0.641116, 0...",0.706712,0.679989,0.604023,0.641116,0.672666,0.636422,0.768434,0.632805,0.706544,0.647260,0.608389,1
397,Unusable wood scarps were discarded in wrong p...,"[0.68631643, 0.64990467, 0.59430903, 0.6379744...",0.686316,0.649905,0.594309,0.637974,0.740876,0.773018,0.660722,0.833772,0.704854,0.726929,0.623767,8


In [76]:
# now filter based on threshold 
min = 0.9
max = 1.0
# drop any rows where the highest score from the 11 scores is not between min and max
all_scores_expanded_filtered = all_scores_expanded[(all_scores_expanded[["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]].max(axis=1) >= min) & (all_scores_expanded[["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]].max(axis=1) <= max)]
all_scores_expanded_filtered

,Input phrase,Similarity scores,1,2,3,4,5,6,7,8,9,10,11,Consensus code
23,lathe chuck tightened too tight,"[0.93225986, 0.7850034, 0.61355317, 0.6118187,...",0.932260,0.785003,0.613553,0.611819,0.710815,0.612184,0.624168,0.591448,0.694529,0.612434,0.620547,1
26,Finding clamps,"[0.6889468, 0.66507596, 0.623487, 0.66982204, ...",0.688947,0.665076,0.623487,0.669822,0.913072,0.678755,0.713313,0.677950,0.689193,0.674919,0.631177,5
31,Height of miter saw,"[0.58212316, 0.6182599, 0.57936734, 0.9281533,...",0.582123,0.618260,0.579367,0.928153,0.603224,0.678420,0.623901,0.606485,0.611006,0.576042,0.561073,4
33,Ripped Trash bag,"[0.6559895, 0.6101904, 0.6384808, 0.6411876, 0...",0.655990,0.610190,0.638481,0.641188,0.700307,0.696051,0.696083,0.727758,0.650865,1.000000,0.562800,10
43,She was bumped into,"[0.65827346, 0.66756225, 0.6125785, 0.68157375...",0.658273,0.667562,0.612579,0.681574,0.733103,0.702337,0.905657,0.629873,0.684305,0.706430,0.634523,7
57,Ripped trash bag,"[0.6559895, 0.6101904, 0.6384808, 0.6411876, 0...",0.655990,0.610190,0.638481,0.641188,0.700307,0.696051,0.696083,0.727758,0.650865,1.000000,0.562800,10
85,Someone bumped into her,"[0.6348177, 0.6499023, 0.6004541, 0.6723578, 0...",0.634818,0.649902,0.600454,0.672358,0.727994,0.703276,0.910152,0.628600,0.674638,0.699084,0.629565,7
86,Trash was ripped,"[0.6610016, 0.645005, 0.64369893, 0.64744973, ...",0.661002,0.645005,0.643699,0.647450,0.729360,0.697385,0.704337,0.759959,0.670853,0.929252,0.594642,10
90,Got bumped into,"[0.659939, 0.6740671, 0.6244695, 0.6828936, 0....",0.659939,0.674067,0.624470,0.682894,0.739855,0.720194,0.937652,0.655540,0.705832,0.724154,0.636733,7
95,Rip in trash bag,"[0.6765851, 0.6544485, 0.641317, 0.6397744, 0....",0.676585,0.654449,0.641317,0.639774,0.731274,0.721945,0.710640,0.735425,0.643936,0.931789,0.598167,10


In [77]:
# now, get cohens kappa score for the filtered data
ground_truths = all_scores_expanded_filtered["Consensus code"].tolist()
predictions = all_scores_expanded_filtered[["1", "2", "3", "4", "5", "6", "7", "8", "9", "10", "11"]].idxmax(axis=1).tolist()
# convert predictions from strings to ints
predictions = [int(x) for x in predictions]
kappa = cohen_kappa_score(ground_truths, predictions, labels=[1,2,3,4,5,6,7,8,9,10,11], weights=None, sample_weight=None)
print(kappa)

1.0
